# OmniProof Playground

**Causal-Multimodal Engine for Creative Performance Attribution**

This notebook walks through every OmniProof capability as a **connected pipeline** — from embedding creatives into Pinecone, through causal analysis, to DICE-DML on A/B video pairs. Each walkthrough builds on the shared clients initialized in the setup cell.

| Walkthrough | What it shows | API keys? |
|:--|:--|:--|
| 0. Offline Demo | All 9 pipeline stages in ~4s | No |
| 1. Embed into Pinecone | Gemini Embedding 2 → Pinecone storage | Yes |
| 2. Extract Metadata | Structured creative metadata via Flash Lite | Yes |
| 3. Brand Identity | Multimodal brand profile extraction | Yes |
| 4. Brand Compliance | RAG: embed guidelines → retrieve → evaluate | Yes |
| 5. Causal Analysis | DAG → DML → ATE/CATE → refutation | No |
| 6. Embeddings + Causal | Merge Pinecone embeddings with campaign data | Yes |
| 7. DICE-DML | Counterfactual pairs → treatment fingerprint → visual ATE | Yes |
| 8. Design Brief | Causal insights → creative prompt generation | No |
| 9. API Server | FastAPI endpoints for all capabilities | Yes |

**Prerequisites:**
- `pip install -e ".[dev]"` from the project root
- `pip install jupyter` if not already installed
- A [Gemini API key](https://aistudio.google.com/apikey) and [Pinecone](https://app.pinecone.io/) account (for walkthroughs 1-4, 6-7, 9)

In [ ]:
import os
from pathlib import Path

# Change to project root so all paths work correctly
PROJECT_ROOT = Path().resolve().parent if Path("examples").exists() is False else Path().resolve()
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

# Verify examples exist
assert Path("examples/creatives").exists(), "examples/creatives not found — check PROJECT_ROOT"
png_count = len(list(Path("examples/creatives").glob("*.png")))
mp4_count = len(list(Path("examples/creatives").glob("*.mp4")))
print(f"Creatives found: {png_count} PNGs, {mp4_count} MP4s")

# --- Fill in your keys here ---
os.environ["OMNI_PROOF_GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY"
os.environ["OMNI_PROOF_PINECONE_API_KEY"] = "YOUR_PINECONE_API_KEY"
os.environ["OMNI_PROOF_PINECONE_INDEX_HOST"] = "YOUR_PINECONE_INDEX_HOST"

# Initialize shared clients used by all walkthroughs
from pinecone import Pinecone
from omni_proof.ingestion.gemini_client import GeminiClient
from omni_proof.storage.vector_store import PineconeVectorStore

GEMINI_KEY = os.environ["OMNI_PROOF_GEMINI_API_KEY"]
client = GeminiClient(api_key=GEMINI_KEY)

pc = Pinecone(api_key=os.environ["OMNI_PROOF_PINECONE_API_KEY"])
index = pc.Index(host=os.environ["OMNI_PROOF_PINECONE_INDEX_HOST"])
store = PineconeVectorStore(index=index)

print(f"Gemini client ready: {bool(GEMINI_KEY)}")
print(f"Pinecone store ready: {bool(store)}")

---
## Walkthrough 0: Offline Demo (No API Keys)

Runs all 9 stages of the OmniProof pipeline using **only local data** — no API keys needed. This is a great sanity check that the install works and a quick overview of what OmniProof does:

1. Campaign overview (1,000 rows across 5 campaigns)
2. Causal DAG construction
3. ATE estimation via Double Machine Learning
4. CATE by audience segment
5. Refutation tests (placebo, subset, random confounder)
6. Design brief synthesis
7. Brand profile loading
8. Creative prompt generation
9. Compliance report samples

In [ ]:
%run examples/demo.py

---
## Walkthrough 1: Embed Creatives into Pinecone

**Requires:** Gemini API key + Pinecone

This walkthrough uses [Gemini Embedding 2](https://ai.google.dev/gemini-api/docs/embeddings) to map all example creatives — images and videos — into a shared **3072-dimensional** semantic space, then stores them in **Pinecone** for retrieval.

Why Pinecone instead of in-memory? In production you need:
- **Persistence** across sessions
- **Scalable search** over millions of embeddings
- **Namespace isolation** (e.g., `"creatives"` vs `"brand_assets"`)

Gemini Embedding 2 supports Matryoshka dimensions (128/768/1536/3072) — we use the full 3072 for maximum fidelity.

In [ ]:
count = 0

async def embed_creatives():
    global count
    creative_dir = Path("examples/creatives")

    # Embed images
    for img in sorted(creative_dir.glob("*.png")):
        embedding = await client.generate_embedding(img)
        await store.upsert(
            asset_id=img.stem,
            embedding=embedding,
            metadata={"file": img.name, "type": "image", "creative_name": img.stem.rsplit("_", 1)[0]},
            namespace="creatives",
        )
        count += 1
        print(f"  Embedded {img.name} -> {len(embedding)} dims")

    # Embed video variants
    for vid in sorted(creative_dir.glob("*.mp4")):
        embedding = await client.generate_embedding(vid)
        await store.upsert(
            asset_id=vid.stem,
            embedding=embedding,
            metadata={"file": vid.name, "type": "video", "creative_name": vid.stem},
            namespace="creatives",
        )
        count += 1
        print(f"  Embedded {vid.name} -> {len(embedding)} dims")

    print(f"\nTotal assets embedded into Pinecone: {count}")

    # Verify: query back one embedding
    test_emb = await client.generate_embedding(creative_dir / "runner_sunrise_1773607608677.png")
    results = await store.search(query_embedding=test_emb, top_k=3, namespace="creatives")
    print(f"\nVerification — top 3 similar to runner_sunrise:")
    for r in results:
        print(f"  {r['id']} (score: {r['score']:.4f})")

await embed_creatives()

---
## Walkthrough 2: Extract Structured Metadata

**Requires:** Gemini API key

The `IngestPipeline` combines two Gemini calls:
1. **Gemini 3.1 Flash Lite** extracts structured metadata (objects detected, CTA type, promotional text, visual composition) into a Pydantic `CreativeMetadata` schema
2. **Gemini Embedding 2** generates the 3072-dim embedding

This metadata is what populates the relational DB (SQL) side of OmniProof, while the embedding goes to Pinecone.

In [ ]:
from omni_proof.ingestion.pipeline import IngestPipeline
from omni_proof.ingestion.schemas import CreativeMetadata

pipeline = IngestPipeline(gemini_client=client)

async def extract():
    asset = Path("examples/creatives/runner_sunrise_1773607608677.png")
    metadata, embedding = await pipeline.ingest(asset, CreativeMetadata)

    print(f"Asset: {asset.name}")
    print(f"Objects detected: {metadata.visual.objects_detected}")
    print(f"CTA type: {metadata.textual.cta_type}")
    print(f"Promo text: '{metadata.textual.promotional_text}'")
    print(f"Embedding dimensions: {len(embedding)}")

await extract()

---
## Walkthrough 3: Extract Brand Identity

**Requires:** Gemini API key + Pinecone

The `BrandExtractor` analyzes a set of creatives to build a structured brand profile:
- **Visual style**: dominant colors, typography, layout patterns
- **Voice**: formality, emotional register, vocabulary themes
- **Rules**: extracted guidelines (color palette, typography, tone, logo)
- **Visual fingerprint**: averaged embedding across all assets (stored in Pinecone)

Unlike walkthrough 1 which just stores embeddings, this uses Gemini Flash Lite for **structured extraction** — analyzing what's *in* the creatives to infer brand guidelines.

In [ ]:
from omni_proof.brand_extraction.extractor import BrandExtractor

extractor = BrandExtractor(embedding_provider=client, gemini_client=client, vector_store=store)

async def extract_brand():
    assets = sorted(Path("examples/creatives").glob("*.png"))
    print(f"Analyzing {len(assets)} creatives...\n")

    profile = await extractor.extract("Velocity Sportswear", assets)

    print(f"Brand: {profile.brand_name}")
    print(f"Dominant colors: {profile.visual_style.dominant_colors}")
    print(f"Typography: {profile.visual_style.typography_styles}")
    print(f"Voice: {profile.voice.formality}, {profile.voice.emotional_register}")
    print(f"\nRules extracted: {len(profile.rules)}")
    for rule in profile.rules:
        print(f"  [{rule.section_type}] {rule.description[:80]}...")

await extract_brand()

---
## Walkthrough 4: Brand Compliance via RAG

**Requires:** Gemini API key + Pinecone

This is the full RAG (Retrieval-Augmented Generation) compliance pipeline:

1. **Index**: Embed all 12 brand guidelines into Pinecone's `"brand_assets"` namespace
2. **Retrieve**: For a given creative, embed it and find the most relevant guidelines via cosine similarity
3. **Evaluate**: Gemini Flash Lite checks the creative against each retrieved guideline, producing a structured `ComplianceReport`

The key insight: we don't check against *all* guidelines (wasteful), only the **semantically relevant** ones. A logo-focused creative triggers logo rules; a copy-heavy creative triggers tone rules.

In [ ]:
import json
from omni_proof.rag.brand_retriever import BrandRetriever
from omni_proof.orchestration.compliance_chain import ComplianceChain

async def check_compliance():
    # Step 1: Index brand guidelines into Pinecone
    with open("examples/data/brand_guidelines.json") as f:
        guidelines = json.load(f)

    for g in guidelines:
        emb = await client.generate_embedding(g["description"], task_type="RETRIEVAL_DOCUMENT")
        await store.upsert(
            asset_id=g["guideline_id"],
            embedding=emb,
            metadata={"source_type": "guideline", "section": g["section"]},
            namespace="brand_assets",
        )
    print(f"Indexed {len(guidelines)} brand guidelines into Pinecone\n")

    # Step 2: Check a creative against the guidelines
    retriever = BrandRetriever(gemini_client=client, vector_store=store)
    chain = ComplianceChain(gemini_client=client, brand_retriever=retriever)
    report = await chain.check_compliance(
        "VEL-2026-Q1-0012",
        Path("examples/creatives/runner_sunrise_1773607608677.png"),
    )

    print(f"Asset: VEL-2026-Q1-0012")
    print(f"Passed: {report.passed}")
    print(f"Score: {report.score}")
    print(f"Violations: {len(report.violations)}")
    print(f"Guidelines retrieved: {report.evidence_sources}")

await check_compliance()

---
## Walkthrough 5: Causal Analysis (Standard DML)

**No API keys needed** — runs entirely on local CSV data.

This is the core of OmniProof's causal engine. Instead of asking *"did creatives with fast pacing get more clicks?"* (correlation), we ask *"does fast pacing **cause** higher CTR, controlling for platform, budget, audience, and region?"*

The pipeline:
1. **DAG** — Define the causal graph: treatment → outcome, with confounders
2. **Identify** — Apply the backdoor criterion to find a valid adjustment set
3. **Estimate** — Double Machine Learning (DML) via EconML: Neyman-orthogonal estimation that's robust to model misspecification
4. **Refute** — Placebo tests, subset validation, random confounder checks

The example CSV has two **planted** causal effects:
- `fast_pacing` → ATE ~+1.9pp CTR (heterogeneous by segment)
- `warm_color_palette` → ATE ~+0.6pp CTR

In [ ]:
import pandas as pd
from omni_proof.causal.dag_builder import CausalDAGBuilder
from omni_proof.causal.identifier import CausalIdentifier
from omni_proof.causal.estimator import DMLEstimator
from omni_proof.causal.refuter import CausalRefuter

# Load and encode data
data = pd.read_csv("examples/data/campaign_performance.csv")
print(f"Loaded {len(data)} rows, {len(data.columns)} columns")

for col in ["platform", "audience_segment", "region", "quarter"]:
    data[col] = data[col].astype("category").cat.codes.astype(float)

confounders = ["platform", "audience_segment", "daily_budget_usd", "region", "quarter"]

# Build DAG and identify causal effect
dag = CausalDAGBuilder()
model = dag.build_dag(data, treatment="fast_pacing", outcome="ctr", confounders=confounders)
CausalIdentifier().identify_effect(model)
print("DAG built and effect identified via backdoor criterion")

In [ ]:
# Estimate ATE (Average Treatment Effect)
estimator = DMLEstimator(cv=3, n_estimators=30)
ate = estimator.estimate_ate(data, "fast_pacing", "ctr", confounders)
print(f"ATE of fast_pacing on CTR: {ate.ate*100:+.1f} percentage points")
print(f"95% CI: [{ate.ci_lower*100:.1f}, {ate.ci_upper*100:.1f}]pp")
print(f"p-value: {ate.p_value:.4f}")
print(f"Significant: {'Yes' if ate.p_value < 0.05 else 'No'}")

In [ ]:
# Estimate CATE (Conditional ATE) by audience segment
cate = estimator.estimate_cate(data, "fast_pacing", "ctr", confounders, "audience_segment")
print("CATE by audience segment:")
for seg, eff in cate.segments.items():
    bar = chr(9608) * int(eff.effect * 100 / 0.2)
    print(f"  Segment {seg}: {eff.effect*100:+.1f}pp  {bar}")

# Refutation tests — do the results hold up?
refuter = CausalRefuter(cv=3)
placebo = refuter.placebo_test(data, "fast_pacing", "ctr", confounders)
subset = refuter.subset_test(data, "fast_pacing", "ctr", confounders)

print(f"\nPlacebo test: {'PASS' if placebo.passed else 'FAIL'} (shuffled effect: {placebo.new_effect*100:.2f}pp vs original: {placebo.original_effect*100:.2f}pp)")
print(f"Subset test:  {'PASS' if subset.passed else 'FAIL'} (80% subset effect: {subset.new_effect*100:.2f}pp vs original: {subset.original_effect*100:.2f}pp)")

---
## Walkthrough 6: Connect Embeddings to Causal Data

**Requires:** Gemini API key + Pinecone (embeddings from Walkthrough 1)

This is the **missing link** between the embedding world and the causal world. In walkthroughs 1-4, embeddings lived in Pinecone. In walkthrough 5, causal analysis ran on tabular CSV data. But the real power comes from **merging** them:

1. Query Pinecone for each creative's embedding
2. Join with the campaign performance CSV by `creative_name`
3. Now each row has both **performance outcomes** (CTR, conversions) and **visual representation** (3072-dim embedding)

This combined dataset is what feeds into DICE-DML (walkthrough 7) — where we estimate causal effects directly from visual features.

In [ ]:
import numpy as np

# Load campaign data (fresh copy without encoding)
campaign_data = pd.read_csv("examples/data/campaign_performance.csv")
unique_creatives = campaign_data["creative_name"].unique()
print(f"Unique creatives in campaign data: {len(unique_creatives)}")

# Query Pinecone for each creative's embedding
async def fetch_embeddings():
    creative_embeddings = {}
    # Use a reference embedding to search
    for name in unique_creatives:
        # Search by creative name in metadata
        # We'll use the stored embeddings from walkthrough 1
        results = await store.search(
            query_embedding=[0.0] * 3072,  # dummy query — we filter by metadata
            top_k=1,
            filters={"creative_name": name},
            namespace="creatives",
        )
        if results:
            creative_embeddings[name] = results[0]
    return creative_embeddings

# Alternative: directly embed the creative files and merge
import json
with open("examples/data/creative_metadata_samples.json") as f:
    metadata_samples = json.load(f)

creative_to_file = {m["creative_name"]: m["file_name"] for m in metadata_samples}

async def build_merged_dataset():
    embeddings_map = {}
    creative_dir = Path("examples/creatives")
    
    for name, filename in creative_to_file.items():
        filepath = creative_dir / filename
        if filepath.exists():
            emb = await client.generate_embedding(filepath)
            embeddings_map[name] = emb
    
    print(f"Retrieved embeddings for {len(embeddings_map)} creatives")
    
    # Build embedding matrix aligned with campaign data
    emb_rows = []
    matched_indices = []
    for idx, row in campaign_data.iterrows():
        cname = row["creative_name"]
        if cname in embeddings_map:
            emb_rows.append(embeddings_map[cname])
            matched_indices.append(idx)
    
    embeddings_matrix = np.array(emb_rows)
    merged = campaign_data.loc[matched_indices].reset_index(drop=True)
    
    print(f"\nMerged dataset: {merged.shape[0]} rows x {merged.shape[1]} tabular columns + {embeddings_matrix.shape[1]} embedding dims")
    print(f"Embedding matrix shape: {embeddings_matrix.shape}")
    print(f"\nThis combined data feeds into DICE-DML (walkthrough 7)")
    
    return merged, embeddings_matrix

merged_data, embeddings_matrix = await build_merged_dataset()

---
## Walkthrough 7: DICE-DML — Causal Effects from Visual Embeddings

**Requires:** Gemini API key + Pinecone

Standard DML (walkthrough 5) works on tabular features. But what if the treatment is a **visual** property — like pacing style — that's entangled with other visual elements (scene composition, lighting, color)?

**DICE-DML** (Disentangled Causal Embeddings) solves this:

1. **Counterfactual pairs**: Take A/B video variants that differ *only* in the treatment (fast vs slow pacing, same scene otherwise)
2. **Treatment fingerprint**: Subtract embeddings (`A - B`) to isolate the pure treatment signal in embedding space
3. **Disentangle**: Project out the treatment signal from all embeddings, separating treatment from confounding visual features
4. **Estimate**: Run DML on the disentangled embeddings to get a clean causal estimate

This is the key innovation — instead of hand-coding visual features as tabular columns, we let the embeddings carry the information and surgically remove confounding.

In [ ]:
from omni_proof.causal.dice_dml.counterfactual_generator import CounterfactualGenerator
from omni_proof.causal.dice_dml.disentangler import TreatmentDisentangler
from omni_proof.causal.dice_dml.visual_estimator import VisualDMLEstimator

async def run_dice_dml():
    creatives = Path("examples/creatives")

    # Step 1: Generate counterfactual pair from A/B video variants
    generator = CounterfactualGenerator(gemini_client=client)
    pair = await generator.generate(
        original_path=creatives / "runner_sunrise_fast_pacing_A.mp4",
        counterfactual_path=creatives / "runner_sunrise_slow_pacing_B.mp4",
        treatment_attr="fast_pacing",
    )
    print(f"Counterfactual pair generated")
    print(f"  Background similarity: {pair.background_similarity:.4f} (should be ~1.0)")

    # Step 2: Extract treatment fingerprint
    disentangler = TreatmentDisentangler()
    fingerprint = disentangler.extract_treatment_fingerprint(
        pair.original_emb, pair.counterfactual_emb
    )
    print(f"  Treatment fingerprint norm: {np.linalg.norm(fingerprint):.4f}")

    # Step 3: Use merged data from walkthrough 6
    # Encode treatment as binary
    treatment = merged_data["fast_pacing"].values.astype(float)
    outcome = merged_data["ctr"].values.astype(float)

    # Step 4: Estimate visual ATE using DICE-DML
    visual_estimator = VisualDMLEstimator(cv=3)
    visual_ate = visual_estimator.estimate_visual_ate(
        embeddings=embeddings_matrix,
        treatment=treatment,
        outcome=outcome,
        treatment_fingerprint=fingerprint,
    )

    print(f"\nDICE-DML Results:")
    print(f"  Visual ATE of fast_pacing on CTR: {visual_ate.ate*100:+.1f}pp")
    print(f"  95% CI: [{visual_ate.ci_lower*100:.1f}, {visual_ate.ci_upper*100:.1f}]pp")
    print(f"  p-value: {visual_ate.p_value:.4f}")
    print(f"  n_samples: {visual_ate.n_samples}")
    print(f"\nCompare with standard DML ATE: {ate.ate*100:+.1f}pp")
    print(f"DICE-DML operates directly on visual embeddings — no manual feature engineering needed.")

await run_dice_dml()

---
## Walkthrough 8: Design Brief + Creative Prompt

**No API keys needed** — uses CATE results from Walkthrough 5.

The `InsightSynthesizer` converts raw causal estimates into **natural-language design briefs** — actionable recommendations for the creative team. Then `GenerativePromptBuilder` combines:
- Causal insights (which treatments work, for which segments)
- Brand rules (from the brand profile)
- Campaign constraints (format, duration, etc.)

...into an optimized prompt ready for Gemini or any generative model.

In [ ]:
import json
from omni_proof.orchestration.insight_synthesizer import InsightSynthesizer
from omni_proof.api.generative_loop import GenerativePromptBuilder

# Synthesize a design brief from CATE results
brief = InsightSynthesizer().synthesize(cate)
print(f"Finding: {brief.finding}")
print(f"Recommendation: {brief.recommendation}")
print(f"Confidence: {brief.confidence}\n")

# Load brand profile for prompt generation
with open("examples/data/brand_profile.json") as f:
    profile = json.load(f)

# Generate a creative prompt
builder = GenerativePromptBuilder()
prompt = builder.build_prompt(
    cate_insights=[{"treatment": "fast_pacing", "effect": 0.025}],
    brand_rules=[{"description": r["description"]} for r in profile["rules"]],
    target_segment="Gen-Z (18-24)",
    objective="conversion",
    constraints=["9:16 vertical", "max 15 seconds", "product reveal by second 5"],
)

print("--- Generated Creative Prompt ---")
print(prompt)

---
## Walkthrough 9: API Server

**Requires:** All API keys set

OmniProof ships a FastAPI server with 12 endpoints covering brand extraction, compliance, causal analysis, and creative generation. Start the server, then test it with the cells below.

In [ ]:
import subprocess, sys, time

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "omni_proof.api.app:create_app", "--factory", "--port", "8000"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(3)

if server.poll() is None:
    print(f"Server started (PID: {server.pid}) on http://localhost:8000")
    print("Run server.terminate() to stop it")
else:
    print("Server failed to start. Stderr:")
    print(server.stderr.read().decode())

In [ ]:
import urllib.request, json

# Health check
resp = urllib.request.urlopen("http://localhost:8000/health")
print("Health:", json.loads(resp.read()))

# Causal analysis endpoint
req = urllib.request.Request(
    "http://localhost:8000/api/v1/causal/analyze",
    data=json.dumps({
        "treatment": "fast_pacing", "outcome": "ctr",
        "confounders": ["platform", "audience_segment", "daily_budget_usd"]
    }).encode(),
    headers={"Content-Type": "application/json"},
)
resp = urllib.request.urlopen(req)
print("\nCausal analysis:", json.dumps(json.loads(resp.read()), indent=2))

In [ ]:
import urllib.request, json

# Creative prompt generation endpoint
req = urllib.request.Request(
    "http://localhost:8000/api/v1/generative/prompt",
    data=json.dumps({
        "target_segment": "Gen-Z (18-24)", "objective": "conversion"
    }).encode(),
    headers={"Content-Type": "application/json"},
)
resp = urllib.request.urlopen(req)
print("Prompt:", json.dumps(json.loads(resp.read()), indent=2))

# Clean up
server.terminate()
print("\nServer stopped.")